# CST Phase 0 — English Multi-Seed

Runs 5 seeds × {CST-8K, SPM-8K, CST-32K, SPM-32K} = **20 training runs** at 10M params, 3 epochs each.

**Runtime:** T4 GPU ≈ 1.5–2 h. A100 ≈ 20–30 min.

## Setup (one-time)

1. Runtime → Change runtime type → **GPU** (T4 is fine).
2. Upload `colab-upload-en.tar.gz` (72 MB) to the Colab file panel, OR mount Drive and copy it.
3. Run cells in order.

## 1. Install deps

In [ ]:
!pip install -q torch transformers

## 2. Extract tarball

Assumes `colab-upload-en.tar.gz` is in the Colab working dir (`/content/`). If you uploaded to Drive, adjust the path.

In [ ]:
import os, pathlib
assert pathlib.Path('/content/colab-upload-en.tar.gz').exists(), \
    'Upload colab-upload-en.tar.gz to /content/ first (left sidebar → Files → Upload)'
!tar -xzf /content/colab-upload-en.tar.gz -C /content/
!ls -lh /content/colab-upload-en/

## 3. GPU check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 4. Run Phase 0 multi-seed

Skipping downstream (LAMBADA files not yet prepared). BPC comparison is the core Phase 0 deliverable.

You can reduce `--seeds` to `0 1 2` for a quicker smoke test first.

In [ ]:
!cd /content/colab-upload-en && python scripts/experiments/colab_phase0.py \
    --data-dir /content/colab-upload-en \
    --out-dir  /content/phase0_out \
    --langs en \
    --runs 8k 32k \
    --seeds 0 1 2 3 4 \
    --epochs 3 \
    --skip-downstream

## 5. Inspect results

In [ ]:
!cat /content/phase0_out/summary.md

In [ ]:
import json
from collections import defaultdict
import math

rows = json.load(open('/content/phase0_out/results_en.json'))
by_name = defaultdict(list)
for r in rows:
    by_name[r['name']].append(r['best_val_bpc'])

print(f'{"Tokenizer":<10} {"mean BPC":>10} {"std":>8} {"n":>4}')
for name, xs in sorted(by_name.items()):
    m = sum(xs)/len(xs)
    v = sum((x-m)**2 for x in xs)/max(len(xs)-1, 1)
    print(f'{name:<10} {m:>10.4f} {math.sqrt(v):>8.4f} {len(xs):>4}')

## 6. Download results

Pull the output dir back to your local machine:

In [ ]:
!tar -czf /content/phase0_out.tar.gz -C /content phase0_out
from google.colab import files
files.download('/content/phase0_out.tar.gz')